# A file responsible for pinging the Visual Crossing API to download the missing data


## Load API Keys,  Location CSVS

In [21]:
import os
from dotenv import load_dotenv

# Load variables from the .env file into the environment
load_dotenv()

# Retrieve the specific key using os.getenv
api_keys = [
    os.getenv("Key0"),
    os.getenv("Key1"),
    os.getenv("Key2"),
    os.getenv("Key3"),
    os.getenv("Key4"),
    os.getenv("Key5"),
    os.getenv("Key6"),
]

if api_keys:
    print(f"✅Successfully loaded {len(api_keys)} API keys.")
else:
    print("❌API Keys not found. Check your .env file.")


✅Successfully loaded 7 API keys.


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

locationData = pd.read_csv("locationdata.csv", encoding='latin1' )
locationData.head(10)

,locationURL,locationCode
0,Port%20of%20Spain%2C%20Trinidad%2C%20Trinidad%...,Port-of-Spain
1,San%20Fernando%2C%20Trinidad%2C%20Trinidad%20a...,San-Fernando
2,Arima%2C%20Trinidad%2C%20Trinidad%20and%20Tobago,Arima
3,Sangre%20Grande%2C%20Trinidad%2C%20Trinidad%20...,Sangre-Grande
4,Tunapuna-Piarco%2C%20Trinidad%2C%20Trinidad%20...,Tunapuna-Piarco
5,Crown%20Point%2C%20Tobago%2C%20Trinidad%20and%...,Crown-Point
6,Scarborough%2C%20Tobago%2C%20Trinidad%20and%20...,Scarborough
7,Rio%20Claro%20-%20Mayaro%2C%20Trinidad%2C%20Tr...,Rio-Claro-Mayaro
8,Siparia%2C%20Trinidad%2C%20Trinidad%20and%20To...,Siparia
9,Chaguanas%2C%20Trinidad%2C%20Trinidad%20and%20...,Chaguanas


In [23]:
# Extract columns into lists
locationUrl = locationData["locationURL"].tolist()
locationCode = locationData["locationCode"].tolist()

In [24]:
print("Ensured location URLs and codes are loaded:")
print(f"📌Location URL: {locationUrl[0]}")
print(f"📌Location Code: {locationCode[0]}")

Ensured location URLs and codes are loaded:
📌Location URL: Port%20of%20Spain%2C%20Trinidad%2C%20Trinidad%20and%20Tobago
📌Location Code: Port-of-Spain


## Define API call

In [25]:
# Constants for API requests
base_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"
unit_group = "us"
include = "days"
content_type = "csv"

The initial call was insufficient to acquire all the desired years. It would download a few and skip over certain years inbetween due to daily download limitations for the API key being used. Therefore, the initial function was modified to take in perameters such as a new API key as well as a list with the missing years for them to be acquired.

In [26]:
import time
import requests

folder_path = "raw_weather_data"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def fetch_weather_data(start_date, end_date, location, loc_code, base_url, key, unit_group, include, content_type):
    for attempt in range(10):
        try:
            url = f"{base_url}{location}/{start_date}/{end_date}?unitGroup={unit_group}&include={include}&key={key}&contentType={content_type}"

            base_folder = f"raw_weather_data/{loc_code}/"
            if not os.path.exists(base_folder):
                os.makedirs(base_folder)

            filename = f"weather_data_{loc_code}_{start_date}_to_{end_date}.csv"
            filepath = os.path.join(base_folder, filename)
            response = requests.get(url)
            
            if response.status_code == 200:
                with open(filepath, "wb") as f:
                    f.write(response.content)
                
                print(f"Saved: {filepath}")
                return True

            if response.status_code == 429 and key == api_keys[-1]:  # If the last key is also exhausted
                print ("❌All API keys have been exhausted. Please wait and try again later.")
                return False
            
            if response.status_code == 429:
                print("⚠️Rate limit exceeded → waiting before retrying...")
                key = api_keys[(api_keys.index(key) + 1) % len(api_keys)]  # Switch to the next API key
                print(f"🆕Switched to new API key number {api_keys.index(key)}")
                time.sleep(5)  # Wait for 5 seconds before retrying
                
            else:
                print(f"❌Failed ({response.status_code}) → retrying...")
                time.sleep(5)
        
        except requests.exceptions.RequestException as e:
            print(f"❌Error: {e}")
            time.sleep(5)

In [27]:
import pandas as pd
import glob
import os

def combine_weather_data(loc_code):
    # 📁 Folder paths
    raw_folder_path = f"./raw_weather_data/{loc_code}/"
    cleaned_folder_path = f"./preprocessed_weather_data/"

    # Ensure output folder exists
    os.makedirs(cleaned_folder_path, exist_ok=True)

    # Step 1: Get all CSV files
    all_files = glob.glob(os.path.join(raw_folder_path, "*.csv"))

    if not all_files:
        print(f"❌ No CSV files found in {raw_folder_path}")
        return

    all_dfs = []

    # Step 2: Load and clean each file
    for file in all_files:
        try:
            df = pd.read_csv(file)

            # Skip if 'datetime' column missing
            if 'datetime' not in df.columns:
                print(f"⚠️ Skipping {file} (no 'datetime' column)")
                continue

            # Remove duplicate header rows inside data
            df = df[df['datetime'] != 'datetime']

            # Only append non-empty DataFrames
            if not df.empty:
                all_dfs.append(df)

        except Exception as e:
            print(f"⚠️ Error reading {file}: {e}")

    # Step 3: Combine safely
    if not all_dfs:
        print("❌ No valid data to combine.")
        return
    elif len(all_dfs) == 1:
        combined_df = all_dfs[0]
    else:
        combined_df = pd.concat(all_dfs, ignore_index=True)

    # Step 4: Parse datetime
    combined_df['datetime'] = pd.to_datetime(combined_df['datetime'], errors='coerce')

    # Step 5: Drop invalid datetime rows
    combined_df = combined_df.dropna(subset=['datetime'])

    # Step 6: Sort
    combined_df = combined_df.sort_values('datetime')
    # If a record exists without values for these three variables, then that record has no significant data
    columns_to_check = ['temp', 'tempmax', 'tempmin']
    # Drop rows where all of these columns have missing values
    combined_df = combined_df.dropna(subset=columns_to_check, how='all')
    # Drop unnecessary columns. Columns with no relevance to Trinidad and Tobago (like 'snow'), columns with recurring values, and columns with excessive missing values.
    combined_df = combined_df.drop(columns=['snow', 'snowdepth', 'preciptype', 'windgust', 'severerisk'])

    # Step 7: Filter years
    combined_df = combined_df[
        (combined_df['datetime'].dt.year >= start_year) &
        (combined_df['datetime'].dt.year <= end_year)
    ]

    # Step 8: Reset index
    combined_df.reset_index(drop=True, inplace=True)

    # Step 9: Save file
    output_file = os.path.join(
        cleaned_folder_path,
        f"{loc_code}_weather_data_{start_date}_{end_date}.csv"
    )

    combined_df.to_csv(output_file, index=False)

    print(f"✅ Combined file saved: {output_file}")

## API Call variables to obtain details

In [28]:
# Define the year range you want to query
start_year = 2025
end_year = 2026
start_date = f"{start_year}-04-27"
end_date = f"{end_year}-04-04"

locationIndex = 1  # Change this index to fetch data for different locations
location = locationUrl[locationIndex]
loc_code = locationCode[locationIndex]
key = api_keys[0]  # Use the first API key for all requests

In [29]:
# Loop through each year in the defined range
for year in range(start_year, end_year + 1):
    start_date = f"{year}-04-27"
    end_date = f"{year+1}-04-04"
    status =fetch_weather_data(start_date, end_date, location, loc_code, base_url, key, unit_group, include, content_type)

    if not status:
        print("❌Data collection failed during the process. See logs for details. Please try again later.")
        break

if status:
    print ("✅Data collection complete. Now combining files...")
    combine_weather_data(loc_code)


⚠️Rate limit exceeded → waiting before retrying...
🆕Switched to new API key number 1
⚠️Rate limit exceeded → waiting before retrying...
🆕Switched to new API key number 2
⚠️Rate limit exceeded → waiting before retrying...
🆕Switched to new API key number 3
⚠️Rate limit exceeded → waiting before retrying...
🆕Switched to new API key number 4
⚠️Rate limit exceeded → waiting before retrying...
🆕Switched to new API key number 5
⚠️Rate limit exceeded → waiting before retrying...
🆕Switched to new API key number 6
❌All API keys have been exhausted. Please wait and try again later.
❌Data collection failed during the process. See logs for details. Please try again later.


In [30]:
combine_weather_data(loc_code)

❌ No CSV files found in ./raw_weather_data/San-Fernando/


All the files stored for the specific location were then retrieved from the appropriate directory, merged in ascending order (2000 - 2025) and then written to a csv.

The data was read in and cleaned before being re-written to the csv